<a href="https://colab.research.google.com/github/cbonnin88/LimeLight/blob/main/Predictive_Analytics_HR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import polars as pl
import plotly.express as px
from google.cloud import bigquery
from google.colab import auth
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

In [2]:
auth.authenticate_user()
project_id = 'limelight-hr-data'
client = bigquery.Client(project=project_id)

In [5]:
query='''
  SELECT * FROM `limelight-hr-data.dbt_limelight_analytics.dim_employee_directory`
'''
df_raw = client.query(query).to_dataframe()

In [6]:
df_people = pl.from_pandas(df_raw)

In [7]:
display(df_people.head())

employee_id,first_name,last_name,gender,office_id,job_title,department,tenure,remote_type,performance,is_manager_flag,hours,country,base_salary,FTE_salary,salary_band,retention_risk_status
i64,str,str,str,i64,str,str,i64,str,str,bool,i64,str,f64,f64,str,str
10006,"""Joséphine""","""Bourgeois""","""Female""",1,"""ESG Strategy""","""ESG Strategy""",6,"""hybrid""","""average""",false,20,"""France""",49549.630841,91666.82,"""L3""","""Stable"""
10337,"""Nicolas""","""Gilson""","""Male""",7,"""ESG Strategy""","""ESG Strategy""",2,"""remote""","""bottom""",false,20,"""Belgium""",49549.630841,91666.82,"""L3""","""Stable"""
10781,"""Paula""","""Legros""","""Female""",7,"""ESG Strategy""","""ESG Strategy""",5,"""hybrid""","""bottom""",false,20,"""Belgium""",49549.630841,91666.82,"""L3""","""Stable"""
10802,"""Rodolfo""","""Jordá""","""Male""",3,"""ESG Strategy""","""ESG Strategy""",5,"""remote""","""bottom""",false,20,"""Spain""",49549.630841,91666.82,"""L3""","""Stable"""
10842,"""Robert""","""Marie""","""Male""",1,"""ESG Strategy""","""ESG Strategy""",3,"""remote""","""average""",false,20,"""France""",49549.630841,91666.82,"""L3""","""Stable"""


# **1. Creating a "Target" variable: Did they leave ?**

In [9]:
df_people = df_people.with_columns([
    pl.when(
        (pl.col('performance')== 'bottom') &
        (pl.col('salary_band') == 'L5') &
        (pl.col('tenure') > 3)
    ).then(1).otherwise(0).alias('attrition_target')
])

display(df_people.head())

employee_id,first_name,last_name,gender,office_id,job_title,department,tenure,remote_type,performance,is_manager_flag,hours,country,base_salary,FTE_salary,salary_band,retention_risk_status,attrition_target
i64,str,str,str,i64,str,str,i64,str,str,bool,i64,str,f64,f64,str,str,i32
10006,"""Joséphine""","""Bourgeois""","""Female""",1,"""ESG Strategy""","""ESG Strategy""",6,"""hybrid""","""average""",false,20,"""France""",49549.630841,91666.82,"""L3""","""Stable""",0
10337,"""Nicolas""","""Gilson""","""Male""",7,"""ESG Strategy""","""ESG Strategy""",2,"""remote""","""bottom""",false,20,"""Belgium""",49549.630841,91666.82,"""L3""","""Stable""",0
10781,"""Paula""","""Legros""","""Female""",7,"""ESG Strategy""","""ESG Strategy""",5,"""hybrid""","""bottom""",false,20,"""Belgium""",49549.630841,91666.82,"""L3""","""Stable""",0
10802,"""Rodolfo""","""Jordá""","""Male""",3,"""ESG Strategy""","""ESG Strategy""",5,"""remote""","""bottom""",false,20,"""Spain""",49549.630841,91666.82,"""L3""","""Stable""",0
10842,"""Robert""","""Marie""","""Male""",1,"""ESG Strategy""","""ESG Strategy""",3,"""remote""","""average""",false,20,"""France""",49549.630841,91666.82,"""L3""","""Stable""",0


# **2. Select Features for the model**

In [13]:
# I excluded ID's and Names since they don't help prediction
features = ['gender','department','tenure','remote_type','performance','FTE_salary','is_manager_flag']
X = df_people.select(features).to_pandas()
y = df_people.select('attrition_target').to_pandas().values.ravel()

# **3. Encode Categorical Data (Turning text into 0's and 1's)**

In [14]:
X_encoded = pd.get_dummies(X,drop_first=True)

# **Training the Model**

In [15]:
X_train,X_test,y_train,y_test = train_test_split(X_encoded,y,test_size=0.2)

# Initialize the Model (Random Forest is great for this type of data)
model = RandomForestClassifier(n_estimators=100,random_state=42)

# Train the model
model.fit(X_train,y_train)

RandomForestClassifier(random_state=42)

# **3. Model Evalution**

In [16]:
y_pred = model.predict(X_test)

# Checking results
print('--- Classificiation Repost ---')
print(classification_report(y_test,y_pred))

--- Classificiation Repost ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2423
           1       1.00      1.00      1.00        77

    accuracy                           1.00      2500
   macro avg       1.00      1.00      1.00      2500
weighted avg       1.00      1.00      1.00      2500



In [18]:
# Get importance scores
importances = model.feature_importances_
feature_names = X_encoded.columns

importance_df = pd.DataFrame({'Feature':feature_names,'Importance':importances}).sort_values('Importance',ascending=False)

In [21]:
fig_imp = px.bar(
    importance_df,
    x='Importance',
    y='Feature',
    orientation='h',
    title='Top 10 Drivers of Employee Churn (Random Forest Importance)',
    labels={'Importance':'Prediction Power (Normalized Importance)'},
    color='Importance',
    color_continuous_scale='viridis'
)

fig_imp.update_layout(
    yaxis_autorange='reversed',
    coloraxis_showscale= False,
    title_x=0.5,
    template='plotly_white'
)
fig_imp.show()